In [ ]:
#import libraries
import pandas as pd
import numpy as np

#inputs are fighter stats, outputs are fighter won or lost
#given fighters history, what will the outcome be for a fight?

#input tables: fighter fight stats, round stats, fights
#output: fighter x has y probability of winning

#read csv first.
csv_path = "../out.csv" 
df = pd.read_csv(csv_path)
df.info


In [ ]:
#rename stats
rename_stats = {
    "knockdowns": "kd",
    "sig_strikes_landed": "ss_l",
    "sig_strikes_attempted": "ss_a",
    "takedowns_landed": "td_l",
    "takedowns_attempted": "td_a",
    "control_seconds": "ctrl_s",
}
df = pd.read_csv(csv_path, parse_dates=["event_date"]).rename(columns=rename_stats)
assert not df.duplicated(["fight_id", "fighter_id"]).any()
assert df.groupby("fight_id")["fighter_id"].nunique().eq(2).all()

df = df.sort_values(["fighter_id", "event_date", "fight_id"]).reset_index(drop=True)

In [ ]:
#calculate historical win rates
#find all fights where winner_id == curr_fighter_id
#get fights that are win/loss not draw/nc
df["decisive"] = df["outcome"].eq("winner")
df["won"] = df["decisive"] & df["fighter_id"].eq(df["winner_id"])
agr = df.groupby(["fighter_id", "event_date"], as_index=False).agg(
    wins=("won", "sum"),
    decisive_fights=("decisive", "sum")
).sort_values(["fighter_id", "event_date"])
agr

In [ ]:
#pre fight totals
grouped = agr.groupby("fighter_id")
#sum all wins divided by decisive fights and shift up one
for col in ["wins", "decisive_fights"]:
    agr[f"prior_{col}"] = grouped[col].transform(lambda x : x.cumsum().shift(fill_value=0))


agr["prior_wr"] = agr["prior_wins"] / agr["prior_decisive_fights"].replace(0, np.nan)
agr.head(10)


In [ ]:
prior_cols = ["fighter_id", "event_date", "prior_wins", "prior_decisive_fights", "prior_wr"]
df = df.merge(agr[prior_cols], on=["fighter_id", "event_date"], how="left", validate="many_to_one")
df.sort_values(["fighter_id", "event_date"])
df

In [ ]:
#historical avg stats
stats_cols = ["kd", "ss_l", "ss_a", "td_l", "td_a", "ctrl_s"]
#in earlier fights, fighters fought multiple times in one event
agr = df.groupby(["fighter_id", "event_date"], as_index=False).agg(
    ss_a=("ss_a", "sum"),
    ss_l=("ss_l", "sum"),
    td_a=("td_a", "sum"),
    td_l=("td_l", "sum"),
    kd=("kd", "sum"),
    kd_count=("kd", "count"),
    ctrl_s=("ctrl_s", "sum"),
    wins=("won", "sum"),
    decisive_fights=("decisive", "sum")
).sort_values(["fighter_id", "event_date"])

grouped = agr.groupby(["fighter_id"])
#create prior columns
for col in stats_cols + ["kd_count"]:
    agr[f"prior_{col}"] = grouped[col].transform(lambda x : x.cumsum().shift(fill_value=0))

prior_cols = [f"prior_{col}" for col in stats_cols + ["kd_count"]]
# grouped = df.groupby(["fighter_id, event_date"])
merge_keys = ["fighter_id", "event_date"]
df = df.drop(columns=prior_cols, errors="ignore").merge(agr[merge_keys + prior_cols], on=merge_keys, how='left', validate="many_to_one")



In [ ]:
df[merge_keys + stats_cols + prior_cols + ["prior_wins"] + ["prior_wr"]]
#stats columns
print(df.select_dtypes(include="number").columns.to_list())

In [ ]:
#add combined stats 
df["ss%"] = df["prior_ss_l"] / df["prior_ss_a"].replace(0, np.nan)
df["td%"] = df["prior_td_l"] / df["prior_td_a"].replace(0, np.nan)
df["kd_avg"] = df["prior_kd"] / df["prior_kd_count"].replace(0, np.nan)
df[:10][:3]


In [ ]:
df.loc[
    df["fighter_name"].str.casefold().eq("song yadong")
].sort_values("event_date", ascending=False).head(1)

In [ ]:
# One row per decisive matchup; fighter 1 is picked by a seeded random swap.
feature_cols = ["ss%", "td%", "kd_avg", "prior_wr"]
keys = ["event_id", "event_date", "fight_id"]

rows = df.loc[
    df["decisive"],
    keys + ["fighter_id", "fighter_name", "won"] + feature_cols,
].sort_values(["fight_id", "fighter_id"]).copy()
rows["side"] = rows.groupby("fight_id").cumcount()

#randomly swap sides in ~half the fights so fighter 1 isn't always the lower ID
fight_ids = rows["fight_id"].unique()
swap_ids = fight_ids[np.random.default_rng(42).random(len(fight_ids)) < 0.5]
swap = rows["fight_id"].isin(swap_ids)
rows.loc[swap, "side"] = 1 - rows.loc[swap, "side"]

matches = (
    rows.loc[rows["side"].eq(0)]
    .merge(
        rows.loc[rows["side"].eq(1)],
        on=keys,
        suffixes=("_f1", "_f2"),
        validate="one_to_one",
    )
    .sort_values(["event_date", "fight_id"])
    .reset_index(drop=True)
)

X = pd.DataFrame({
    f"{col}_diff": matches[f"{col}_f1"] - matches[f"{col}_f2"]
    for col in feature_cols
})
y = matches["won_f1"].astype(int)  # 1 = fighter 1 wins; 0 = fighter 2 wins. 
assert 0.45 < y.mean() < 0.55, f"side swap broken: y mean {y.mean():.3f}"
y.value_counts().sort_index()
#count 1s and 0s here


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# train = matches["event_date"] < pd.Timestamp("2024-01-01")
#make train 70, val 20, test 10 by event date
dates = matches["event_date"]
val_start, test_start = dates.quantile([0.7, 0.9])

train = dates < val_start
val = (dates < test_start) & (dates >=val_start)
test = dates >=test_start

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X.loc[train], y.loc[train])




In [ ]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score, confusion_matrix

val_probs = model.predict_proba(X.loc[val])[:, 1]
y_val = y.loc[val]
pred_val = (val_probs >=0.5).astype(int)
print("accuracy score = ", accuracy_score(y_val, pred_val))
print("log loss = ", log_loss(y_val, val_probs))
print("brier:   ", brier_score_loss(y_val, val_probs))
print("roc auc: ", roc_auc_score(y_val, val_probs))
print(confusion_matrix(y_val, pred_val))

In [ ]:
y.loc[val].value_counts(normalize=True)

In [ ]:
# song_umar prediction
song_umar = matches.loc[matches["fight_id"].eq(78)]
if len(song_umar) != 1:
    raise ValueError("Expected exactly one matchup with fight_id 78")

song_umar_probs = model.predict_proba(X.loc[song_umar.index])
# Probability columns follow model.classes_, not the fighter order.
class_probs = dict(zip(model.classes_, song_umar_probs[0]))
fight = song_umar.iloc[0]

prediction = pd.DataFrame({
    "fighter_name": [fight["fighter_name_f1"], fight["fighter_name_f2"]],
    "win_probability": [class_probs[1], class_probs[0]],
})
prediction